In [1]:
import pandas as pd, numpy as np
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 25)
df = pd.read_parquet("../out/ksa_sales_raw.parquet")
df.shape

(1269512, 17)

In [2]:
import os; print(os.getcwd())

/Users/hassanmacm1/projects/ksa-house-prices/notebooks


In [3]:
def profile(frame):
    return pd.DataFrame({
        "dtype":      frame.dtypes,
        "nulls":      frame.isna().sum(),
        "null_pct":   (frame.isna().mean() * 100).round(1),
        "nunique":    frame.nunique(),
        "uniq_ratio": (frame.nunique() / len(frame)).round(3),
    })

def red_flags(frame, ignore=("source_file", "quarter"), id_ratio=0.99):
    n = len(frame)
    cols = [c for c in frame.columns if c not in ignore]
    return {
        "constant_cols": frame.columns[frame.nunique(dropna=False) == 1].tolist(),
        "id_like":       [c for c in frame.columns
                          if frame[c].nunique() / n > id_ratio],
        "dup_rows":      int(frame[cols].duplicated().sum()),
        "high_null":     frame.columns[frame.isna().mean() > 0.30].tolist(),
    }

In [4]:
profile(df) 

,dtype,nulls,null_pct,nunique,uniq_ratio
quarter,object,0,0.0,23,0.000
date_greg,datetime64[ns],12,0.0,2006,0.002
date_hijri,object,44103,3.5,1916,0.002
region,object,12,0.0,16,0.000
city,object,12,0.0,174,0.000
city_district,object,44103,3.5,13072,0.010
district,object,1225421,96.5,4091,0.003
plan,object,1225421,96.5,4886,0.004
parcel_no,object,1225421,96.5,12046,0.009
classification,object,12,0.0,6,0.000


In [5]:
profile(df) 

,dtype,nulls,null_pct,nunique,uniq_ratio
quarter,object,0,0.0,23,0.000
date_greg,datetime64[ns],12,0.0,2006,0.002
date_hijri,object,44103,3.5,1916,0.002
region,object,12,0.0,16,0.000
city,object,12,0.0,174,0.000
city_district,object,44103,3.5,13072,0.010
district,object,1225421,96.5,4091,0.003
plan,object,1225421,96.5,4886,0.004
parcel_no,object,1225421,96.5,12046,0.009
classification,object,12,0.0,6,0.000


In [6]:
red_flags(df)

{'constant_cols': [],
 'id_like': [],
 'dup_rows': 44842,
 'high_null': ['district',
  'plan',
  'parcel_no',
  'property_type',
  'price_per_m2']}

In [7]:
# 1 — the 12 rows that are empty in every column
df[df.price_sar.isna()]

# 2 — the region spellings
df.region.value_counts()

# 3 — date coverage per file
df.groupby("quarter").date_greg.agg(["min", "max", "count"])

,min,max,count
quarter,,,
2020-Q1,2020-01-01,2020-03-31,80634
2020-Q2,2020-04-01,2020-06-30,49618
2020-Q3,2020-07-01,2020-09-30,64316
2021-Q2,2021-04-01,2021-06-30,66671
2021-Q3,2021-07-01,2021-09-30,55911
2021-Q4,2021-10-01,2021-12-31,66338
2022-Q1,2022-01-01,2022-03-31,71251
2022-Q2,2022-04-01,2022-06-30,53016
2022-Q3,2022-07-01,2022-09-30,45976


In [8]:
a = df[df.quarter == "2024-Q2"]
b = df[df.quarter == "2025-Q2"]
b.deal_ref.isin(set(a.deal_ref)).sum()

np.int64(47211)

Finding. MOJ-Sales-2025-Q2.csv contains April–June 2024 data, not 2025-Q2. 47,211 of its 47,518 transaction references already appear in MOJ-Sales-2024-Q2.csv (99.4%). Consequences: the dataset has three missing quarters (2020-Q4, 2021-Q1, 2025-Q2), not two; 2024-Q2 is double-weighted; and 44,831 of the 44,842 duplicate rows come from this one file.